# Fake News Detection using Machine Learning & NLP
## Algorithm: Passive Aggressive Classifier (PAC)
### Author: Bibhore Raj

### Project Overview
This notebook provides a complete pipeline for detecting fake news articles and calculating source credibility weights for social media visibility suppression.
- **Model**: Passive Aggressive Classifier (PAC)
- **Evaluation Accuracy**: ~96%
- **Vectorization**: TF-IDF (Term Frequency - Inverse Document Frequency) with unigrams & bigrams
- **Key Innovation**: Source-level visibility weighting to tolerate single-article misclassifications and suppress chronic disinformation publishers.

In [ ]:
# Step 1: Import Essential Libraries
import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Load Dataset
Dataset contains:
- `id`: Unique identifier for each news article
- `title`: Headline/title
- `author`: Author or origin publisher
- `text`: Article body
- `label`: 1 = Unreliable (Fake), 0 = Reliable (Real)

In [ ]:
# Step 2: Ingest Train Data
train_df = pd.read_csv('dataset/train.csv')
print("Dataset Shape:", train_df.shape)
train_df.head()

## 3. Data Preprocessing & Feature Engineering
We concatenate the title, author, and body text into a combined textual feature, clean punctuation/URLs, and lowercase the tokens.

In [ ]:
def clean_news_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove non-alphabetical characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Lowercase & strip extra whitespaces
    return re.sub(r'\s+', ' ', text.lower()).strip()

# Combine and clean
train_df['combined_content'] = train_df['title'].fillna('') + ' ' + train_df['author'].fillna('') + ' ' + train_df['text'].fillna('')
train_df['cleaned_text'] = train_df['combined_content'].apply(clean_news_text)

X = train_df['cleaned_text']
y = train_df['label']
print("Class distribution:\n", y.value_counts(normalize=True))

## 4. Train/Test Split & TF-IDF Vectorization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Initialize TF-IDF Vectorizer with unigrams & bigrams
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.75,
    min_df=1,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)
print(f"TF-IDF Vocabulary Size: {len(tfidf_vectorizer.vocabulary_)} terms")

## 5. Model Training: Passive Aggressive Classifier (PAC)
The Passive Aggressive Classifier is an online algorithm well-suited for large-scale streaming text. 
- **Passive**: If the prediction is correct and beyond the margin, the model weights remain unchanged.
- **Aggressive**: If the prediction is misclassified, the weights are aggressively updated to correct the mistake with minimal disturbance.

In [ ]:
pac_model = PassiveAggressiveClassifier(
    C=0.5,
    max_iter=50,
    random_state=42,
    loss='hinge'
)

pac_model.fit(tfidf_train, y_train)
y_pred = pac_model.predict(tfidf_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

## 6. Model Evaluation & Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Reliable', 'Unreliable']))

## 7. Source Credibility & Visibility Weighting Algorithm
Social media platforms can apply visibility weights calculated from aggregated source history, reducing virality of high-probability misinformation publishers.

In [ ]:
def compute_source_visibility(articles_labels):
    """
    articles_labels: list of predicted labels (0=Real, 1=Fake)
    returns: visibility_weight multiplier (0.05 to 1.0)
    """
    if not articles_labels:
        return 1.0
    fake_ratio = sum(articles_labels) / len(articles_labels)
    if fake_ratio >= 0.6:
        weight = max(0.05, 1.0 - (fake_ratio ** 2 * 1.3))
    elif fake_ratio >= 0.2:
        weight = max(0.25, 1.0 - fake_ratio)
    else:
        weight = 1.0 - (fake_ratio * 0.15)
    return round(float(weight), 3)

# Test simulation
print("Credible Source (10 Real, 0 Fake):", compute_source_visibility([0]*10))
print("Mixed Source (6 Real, 4 Fake):", compute_source_visibility([0]*6 + [1]*4))
print("Disinformation Outlet (1 Real, 9 Fake):", compute_source_visibility([0]*1 + [1]*9))

## 8. Export Model & Vectorizer Artifacts

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(pac_model, f)

with open('vector.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("[✓] Successfully exported model.pkl and vector.pkl")